<a href="https://colab.research.google.com/github/ChristineHarvey/InformationLossScore/blob/main/multiclinsum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


MULTICLINSUM INFORMATION LOSS ANALYSIS SCORE


End-to-end NLP pipeline for measuring information loss between Full Clinical Reports and their associated Clinical Summaries, using the MultiClinSum v8 dataset.


PIPELINE


1. Installation and imports
2. Load models
3. Data ingestion and preprocessing
4. Clinical NER extraction
5. Embedding generation
6. Metric computation and Final Information Loss Score
7. Analysis
8. Visualisation
9. Save results


EXPECTED DATASET STRUCTURE


project_folder/

    multiclinsum_large-scale_train_en/
        fulltext/
            multiclinsum_ls_en_1.txt
            ...

        summaries/
            multiclinsum_ls_en_1_sum.txt
            ...

    multiclinsum_gs_train_en/
        fulltext/
        summaries/

    multiclinsum_test_en/
        fulltext/
        summaries/

In [ ]:
#mount folder
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)
!cp -r "/content/drive/MyDrive/MultiClinSum/" /content/


Mounted at /content/drive/


In [ ]:
#====================================================================
# 1 - INSTALLATION
#====================================================================
!pip install transformers==4.30.0

!pip install pandas numpy matplotlib scikit-learn scipy
!pip install transformers sentence-transformers
!pip install spacy rouge-score bert-score
#!pip install knife-mi-estimator

# ==================================================================
# 1 - IMPORTS
# ==================================================================

import spacy

import os
#import glob
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#from collections import Counter

#from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import normalized_mutual_info_score

from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy

from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer


# ==================================================================
# 2 - LOAD NLP MODELS
# ==================================================================

print("Loading NLP models...")

#Download spaCy model:
!python -m spacy download en_core_web_sm
#OPTIONAL CLINICAL MODEL:
#!pip install scispacy

# spaCy
nlp = spacy.load("en_core_web_sm")

# sentence embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Models loaded")


In [ ]:
#mount folder
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

!cp -r "/content/drive/MyDrive/MScBackupMay/MultiClinSum/" /content/

# ==================================================================
# 3 - DATA INGESTION
# ==================================================================

    #"large_scale": "multiclinsum_large-scale_train_en",
DATASET_FOLDERS = {

    "gold_standard": "/content/MultiClinSum/multiclinsum_gs_train_en",
  #  "test": "/content/MultiClinSum/multiclinsum_test_en"
}


# ------------------------------------------------------------------
# 3 - LOAD DATASETS
# ------------------------------------------------------------------

def load_multiclinsum_dataset(base_folder, dataset_name):
    """
    Loads report-summary pairs from MultiClinSum
    """
    fulltext_folder = os.path.join(base_folder, "fulltext")
    summary_folder = os.path.join(base_folder, "summaries")

    data = []

    report_files = glob.glob(os.path.join(fulltext_folder, "*.txt"))

    print(f"Loading dataset: {dataset_name}")
    print(f"Found {len(report_files)} reports")

    for report_path in report_files:
        report_filename = os.path.basename(report_path)
        # remove .txt
        report_id = report_filename.replace(".txt", "")
        # create matching summary filename
        summary_filename = f"{report_id}_sum.txt"
        summary_path = os.path.join(summary_folder, summary_filename)

        # skip if missing summary
        if not os.path.exists(summary_path):
            print(f"WARNING: Missing summary -> {report_id}")
            continue

        # read report
        with open(report_path, "r", encoding="utf-8") as f:
            report_text = f.read()

        # read summary
        with open(summary_path, "r", encoding="utf-8") as f:
            summary_text = f.read()

        data.append({
            "dataset": dataset_name,
            "report_id": report_id,
            "report_text": report_text,
            "summary_text": summary_text
        })
    return pd.DataFrame(data)


all_dfs = []

for dataset_name, folder_path in DATASET_FOLDERS.items():
    if os.path.exists(folder_path):
        dataset_df = load_multiclinsum_dataset(folder_path, dataset_name)
        all_dfs.append(dataset_df)
    else:
        print(f"WARNING: Folder not found -> {folder_path}")

# combine
df = pd.concat(all_dfs, ignore_index=True)

print("================================================")
print("DATASET OVERVIEW")
print("================================================")

print(df.head())

print("Total samples:", len(df))
print("Dataset counts:")
print(df["dataset"].value_counts())


# ==================================================================
# 3 - DATA PREPROCESSING
# ==================================================================

def preprocess_text(text):
    """
    Basic preprocessing
    """
    text = str(text)
    # lowercase
    text = text.lower()
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    # remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.strip()

print("Preprocessing text...")

df["report_clean"] = df["report_text"].apply(preprocess_text)
df["summary_clean"] = df["summary_text"].apply(preprocess_text )

print("Preprocessing complete.")


In [ ]:
# ==================================================================
# 4 - CLINICAL NER EXTRACTION
# ==================================================================

def extract_entities(text):
    """
    Extract named entities using spaCy
    """
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        entities.append(ent.text.lower())
    return list(set(entities))

print("\nExtracting entities...")

df["report_entities"] = df["report_clean"].apply(extract_entities)
df["summary_entities"] = df["summary_clean"].apply(extract_entities)

print("NER extraction complete.")


# ==================================================================
# 5 - EMBEDDING GENERATION
# ==================================================================

print("\nGenerating embeddings...")

report_embeddings = embedding_model.encode(
    df["report_clean"].tolist(),
    show_progress_bar=True
)

summary_embeddings = embedding_model.encode(
    df["summary_clean"].tolist(),
    show_progress_bar=True
)

print("Embeddings generated.")

In [ ]:

# ==================================================================
# 6 - METRIC COMPUTATION
# ==================================================================

# ------------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------------

print("Computing ROUGE-L")

rouge_scorer_model = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []

for report, summary in zip(df["report_clean"], df["summary_clean"]):
    score = rouge_scorer_model.score(report, summary)
    rouge_scores.append(score["rougeL"].fmeasure)
df["ROUGE_L"] = rouge_scores

# ------------------------------------------------------------------
# BERTScore
# ------------------------------------------------------------------

print("Computing BERTScore")

P, R, F1 = bertscore(
    df["summary_clean"].tolist(),
    df["report_clean"].tolist(),
    lang="en",
    verbose=True
)
df["BERTScore"] = F1.numpy()

# ------------------------------------------------------------------
# ENTITY RECALL + ENTITY LOSS
# ------------------------------------------------------------------

def compute_entity_recall(report_entities, summary_entities):
    report_set = set(report_entities)
    summary_set = set(summary_entities)
    if len(report_set) == 0:
        return 0
    overlap = report_set.intersection(summary_set)
    return len(overlap) / len(report_set)

entity_recalls = []

for report_entities, summary_entities in zip(df["report_entities"], df["summary_entities"]):
    recall = compute_entity_recall(report_entities, summary_entities)
    entity_recalls.append(recall)
df["Entity_Recall"] = entity_recalls

# NB information loss
df["NER_Loss"] = 1 - df["Entity_Recall"]


# ------------------------------------------------------------------
# JENSEN-SHANNON DIVERGENCE
# ------------------------------------------------------------------

print("Computing JS divergence")

vectorizer = CountVectorizer(stop_words="english")

def compute_js_divergence(text1, text2):
    vectors = vectorizer.fit_transform( [text1, text2] )
    v1 = vectors.toarray()[0] + 1e-10
    v2 = vectors.toarray()[1] + 1e-10
    p = v1 / np.sum(v1)
    q = v2 / np.sum(v2)
    return jensenshannon(p, q)

js_scores = []

for report, summary in zip(df["report_clean"], df["summary_clean"]):
    js = compute_js_divergence(report, summary)
    js_scores.append(js)
df["JS_Divergence"] = js_scores


# ------------------------------------------------------------------
# ENTROPY LOSS
# ------------------------------------------------------------------

print("Computing entropy loss")

def compute_entropy(text):
    words = text.split()
    counts = Counter(words)
    probabilities = np.array( list(counts.values()) )
    probabilities = probabilities / probabilities.sum()
    return entropy(probabilities)

entropy_losses = []

for report, summary in zip( df["report_clean"], df["summary_clean"] ):
    report_entropy = compute_entropy(report)
    summary_entropy = compute_entropy(summary)
    if report_entropy == 0:
        entropy_losses.append(0)
    else:
        loss = 1 - ( summary_entropy / report_entropy  )
        entropy_losses.append(loss)
df["Entropy_Loss"] = entropy_losses


# ------------------------------------------------------------------
# EMBEDDING COSINE SIMILARITY
# ------------------------------------------------------------------

print("Computing embedding cosine similarity")

cosine_scores = []

for report_emb, summary_emb in zip(report_embeddings, summary_embeddings):
    similarity = cosine_similarity([report_emb], [summary_emb])[0][0]
    cosine_scores.append(similarity)
df["Embedding_Cosine"] = cosine_scores


# ------------------------------------------------------------------
# Estimate mutual information between two embeddings
# ------------------------------------------------------------------

print("Computing mutual-information loss")

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

def compute_mutual_information(report_embeddings, summary_embeddings):
    # Convert to numpy arrays
    x = np.array(report_embeddings)
    y = np.array(summary_embeddings)

    # Reshape
    x = x.reshape(1, -1)
    y = y.reshape(-1)

    # Standardise
    scaler = StandardScaler()
    x = scaler.fit_transform(x.T).T

    # Estimate MI
    mi = mutual_info_regression( X=x.T, y=y, discrete_features=False  )

    # Mean MI across dimensions
    return np.mean(mi)


# ------------------------------------------------------------
# Mutual information estimator
# ------------------------------------------------------------

# def compute_mutual_information(report_embeddings, summary_embeddings):
#     x = np.array(report_embeddings).reshape(-1, 1)
#     y = np.array(summary_embeddings).reshape(-1, 1)
#     return mi(x, y)

mi_losses = []

for report_emb, summary_emb in zip(report_embeddings, summary_embeddings):
  mi_score = compute_mutual_information(report_emb, summary_emb)
  # Convert MI into loss score, higher MI = lower information loss
  info_loss = np.exp(-mi_score)
  mi_losses.append(info_loss)
df["MI_Information_Loss"] = mi_losses


# # ------------------------------------------------------------------
# # NMI LOSS
# # ------------------------------------------------------------------

# print("Normalise Mutual Information")

# nmi_scores = []
# info_loss_percentages = []

# for report, summary in zip( df["report_clean"], df["summary_clean"] ):
#     nmi = normalized_mutual_info_score(report, summary)
#     nmi_scores.append(nmi)
#     # Calculate relative information loss
#     info_loss_perc = (1 - nmi) * 100
#     info_loss_percentages.append(info_loss_perc)
# df["NMI"] = nmi_scores
# df["IL%"] = info_loss_percentages


# ==================================================================
# 6. FINAL INFORMATION LOSS SCORE
# ==================================================================

print("Computing final Information Loss Score")

"""
Weighted composite score, Adjust weights experimentally later
"""

alpha = 0.35   # NER loss
beta = 0.25    # JS divergence
gamma = 0.20   # entropy loss
delta = 0.20   # embedding semantic loss

df["Information_Loss_Score"] = (
    alpha * df["NER_Loss"]
    + beta * df["MI_Information_Loss"]
    + gamma * (1 - df["BERTScore"])
)

# ctrl /
df["Clinical_Information_Loss"] = (
    alpha * df["MI_Information_Loss"]
    + beta * (1 - df["BERTScore"])
    + gamma * df["NER_Loss"]
)


In [ ]:

# ==================================================================
# 7 - ANALYSIS
# ==================================================================

#convert ROUGE, BERT and cosine into differences instead of similarities
df['ROUGEdiff'] = 1- df['ROUGE_L']
df['BERTdiff'] = 1- df['BERTScore']
df['Cosinediff'] = 1- df['Embedding_Cosine']
print("SUMMARY STATISTICS")
metrics = [
    "ROUGEdiff",
    "BERTdiff",
    "Cosinediff",
    "NER_Loss",
    "JS_Divergence",
    "Entropy_Loss",
    "MI_Information_Loss",
    "Information_Loss_Score"
]
summary_stats = df[metrics].describe()
print(summary_stats)
summary_stats.to_csv('summary_statistics.csv')


# CORRELATION ANALYSIS

print("CORRELATION MATRIX")
correlation_matrix = df[metrics].corr()
print(correlation_matrix)


# DETECT HIGH-LOSS SUMMARIES

print("THRESHOLD ANALYSIS")
threshold = df["Information_Loss_Score"].quantile(0.90)
print("High-loss threshold:", threshold)
df["Poor_Summary"] = (df["Information_Loss_Score"] > threshold)
print("Poor summaries detected:", df["Poor_Summary"].sum())


# ==================================================================
# 8 - VISUALISATIONS
# ==================================================================


# CORRELATION HEATMAP

plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix)
plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45
)
plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)
plt.colorbar()
plt.title("Metric Correlation Matrix")
plt.tight_layout()
plt.show()


# alternative in seaborn
import seaborn as sns
plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,          # show values in cells
    fmt=".2f",           # 2 decimal places
    cmap="coolwarm",
    vmin=0.2,            # lower limit
    vmax=0.8,            # upper limit
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8}
)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.title("Metric Correlation Matrix")
plt.tight_layout()
plt.show()


# BERTScore vs ROUGE-L scores AND Entity Recall
# These are how similar the full reports and summaries are

plt.figure(figsize=(8, 5))
poor = df[df["Poor_Summary"] == True]
plt.scatter(df["BERTScore"], df["ROUGE_L"], color="b", label="ROUGE-L Score")
plt.scatter(df["BERTScore"], df["Entity_Recall"], color="r", label="Entity Recall")
# Highlight poor summaries with black crosses
plt.scatter(poor["BERTScore"], poor["ROUGE_L"], color="yellow", marker="x", s=100, label="Poor Summary (ROUGE-L)")
plt.scatter(poor["BERTScore"], poor["Entity_Recall"], color="black", marker="x", s=100, label="Poor Summary (Entity Recall)")
plt.xlabel("BERTScore")
plt.ylabel("Score")
plt.title("Lexical overlap (ROUGE-L) vs Semantic Similarity (BERTscore)")
plt.legend()
plt.show()


# BOXPLOT

plt.figure(figsize=(6, 5))
plt.boxplot([
    df[ df["Poor_Summary"] == False ]["Information_Loss_Score"],
    df[ df["Poor_Summary"] == True ]["Information_Loss_Score"]
])
plt.xticks([1, 2], ["Good", "Poor"])
plt.ylabel("Information Loss Score")
plt.title("Good vs Poor Summaries")
plt.show()


# HISTOGRAM of BOTH my Information Loss Score and MI

plt.figure(figsize=(8, 5))
plt.hist(df["Information_Loss_Score"], bins=30, color="r", label="Information Loss", alpha=0.5 )
plt.hist(df["MI_Information_Loss"], bins=30, color="b", label="MI Loss", alpha=0.5 )
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("MI vs Proposed Information Loss Score")
plt.legend()
plt.show()


# HISTOGRAM of two different information loss scores

plt.figure(figsize=(8, 5))
plt.hist(df["MI_Information_Loss"], bins=30, color="b", label="Without MI", alpha=0.5 )
plt.hist(df["Clinical_Information_Loss"], bins=30, color="r", label="With MI", alpha=0.5 )
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Comparison of Two Possible Information Loss Scores")
plt.legend()
plt.show()


# ==================================================================
# 9 - SAVE RESULTS
# ==================================================================

OUTPUT_FILE = "multiclinsum_information_loss_results.csv"
df.to_csv(OUTPUT_FILE, index=False)
print("RESULTS SAVED")
print(OUTPUT_FILE)
print("Pipeline complete.")
